In [ ]:
from pathlib import Path
from datetime import datetime
import os
import sys
project_root = Path('c:/PdM/src/data_simulation').resolve()
sys.path.insert(0, str(project_root))
from gas_turbine import GasTurbine
from physics.weather_api_client import create_hybrid_environment
from physics.environmental_conditions import EnvironmentalConditions, LocationType
from ml_utils.ml_output_modes import OutputMode
from datetime import datetime, timedelta

# Create hybrid environment with real weather and synthetic fallback
fallback = EnvironmentalConditions(LocationType.TROPICAL)
env_source = create_hybrid_environment(
    use_real_weather=True,
    api_provider="weatherapi",
    api_key = os.getenv('WEATHER_API_KEY'),
    location_name="Port Harcourt",  # Nigerian coastal installation
    country="Nigeria",
    fallback_source=fallback,  # Falls back to synthetic Tropical on API failure
    cache_enabled=True
)

turbine = GasTurbine(
    name='GT-PH-001',
    initial_health={'hgp': 0.85, 'blade': 0.90, 'bearing': 0.80, 'fuel': 0.88},
    # Pass real weather directly via env_model parameter
    env_model=env_source,  # Real weather from Port Harcourt, Nigeria
    enable_enhanced_vibration=True,
    enable_thermal_transients=True,
    enable_environmental=True,
    enable_maintenance=True,
    enable_incipient_faults=True,
    enable_process_upsets=True,
    output_mode=OutputMode.FULL
)

turbine.set_speed(9500)

# Simulate 30 days with real weather
start_time = datetime(2026, 1, 1)
for hour in range(30 * 24):  # 30 days, hourly
    timestamp = start_time + timedelta(hours=hour)

    # Real weather is automatically fetched and cached
    state = turbine.next_state()

    # Log real weather conditions
    if hour % 24 == 0:  # Daily log
        day = hour // 24
        print(f"Day {day}: Temp={state.get('ambient_temp', 'N/A')}°C, "
              f"EGT={state['exhaust_gas_temp']:.1f}°C, "
              f"Efficiency={state['efficiency']:.3f}")

In [ ]:
from gas_turbine import generate_turbine_dataset

telemetry, failures = generate_turbine_dataset(
    n_machines=5,              # 10 turbines
    n_cycles_per_machine=100,   # 100 operating cycles each
    cycle_duration_range=(60, 300),  # 1-5 minutes per cycle
    random_seed=42
)

print(f"Generated {len(telemetry)} telemetry records")
print(f"Captured {len(failures)} failures")

# Failure distribution
for failure in failures:
    print(f"{failure['machineID']}: {failure['code']} - {failure['message']}")

In [ ]:
from gas_turbine import GasTurbine
from ml_utils.ml_output_modes import OutputMode
import pandas as pd

# Training data (with ground-truth health)
turbine_train = GasTurbine('GT-Train', output_mode=OutputMode.DERIVED_FEATURES)
train_data = []

for i in range(10000):
    state = turbine_train.next_state()
    # state includes 'health_hgp', 'health_blade', etc.
    train_data.append(state)

train_data = pd.DataFrame(train_data)

train_data

In [ ]:
from pathlib import Path
import sys
project_root = Path('c:/PdM/src/data_simulation').resolve()
sys.path.insert(0, str(project_root))
from gas_turbine import GasTurbine
from physics.weather_api_client import create_hybrid_environment
from physics.environmental_conditions import EnvironmentalConditions, LocationType
from ml_utils.ml_output_modes import OutputMode
from datetime import datetime, timedelta
from ml_utils.ml_output_modes import OutputMode
import pandas as pd
turbine_test = GasTurbine('GT-Test', output_mode=OutputMode.SENSOR_ONLY)
turbine_test.set_speed(9500)  # ← Add this!

test_data = []
for i in range(100):
    state = turbine_test.next_state()
    test_data.append(state)

df_test = pd.DataFrame(test_data)
print("Columns:", df_test.columns.tolist())
print("Has health columns?", any('health' in col for col in df_test.columns))


In [ ]:
from pathlib import Path
import sys
project_root = Path('c:/PdM').resolve()
sys.path.insert(0, str(project_root))

from src.data_simulation.gas_turbine import GasTurbine
from src.data_simulation.ml_utils.ml_output_modes import OutputMode


turbine_debug = GasTurbine('GT-Debug', output_mode=OutputMode.DERIVED_FEATURES)
turbine_debug.set_speed(9500)

# Check if formatter was initialized
print(f"use_output_formatter: {turbine_debug.use_output_formatter}")
print(f"Has output_formatter attr: {hasattr(turbine_debug, 'output_formatter')}")

if hasattr(turbine_debug, 'output_formatter'):
    print(f"Output mode: {turbine_debug.output_formatter.output_mode}")
    
    # Test formatter directly
    test_state = turbine_debug.next_state()
    print(f"\nState keys from next_state(): {list(test_state.keys())}")
    print(f"Has health in state: {any('health' in k for k in test_state.keys())}")


turbine_train = GasTurbine('GT-Train', output_mode=OutputMode.DERIVED_FEATURES)
train_data = []

for i in range(10000):
    state = turbine_train.next_state()
    # state includes 'health_hgp', 'health_blade', etc.
    train_data.append(state)

train_data = pd.DataFrame(train_data)




In [9]:
train_data.columns.tolist()

['speed',
 'speed_target',
 'exhaust_gas_temp',
 'oil_temp',
 'fuel_flow',
 'vibration_rms',
 'vibration_peak',
 'compressor_discharge_temp',
 'compressor_discharge_pressure',
 'ambient_temp',
 'ambient_pressure',
 'efficiency',
 'operating_hours',
 'health_hgp',
 'health_blade',
 'health_bearing',
 'health_fuel',
 'num_active_faults',
 'total_faults_initiated',
 'upset_active',
 'features']